<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/01_Raw_Dataset_Loading_%26_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# NOTEBOOK 01 — RAW DATASET LOADING & VALIDATION
# ==============================================================================
#
# Research Project:
#   Privacy-Preserving Synthetic Tabular Data Generation with GANs
#
# Proposed Model:
#   SPP-GAN
#
# Notebook Role:
#   Raw Dataset Loading, Structural Validation, Provenance and Fingerprinting
#
# Scope:
#   01. Header & Scope
#   02. Import Libraries
#   03. Load Notebook 00 Configuration
#   04. Verify Project Structure
#   05. Resolve Dataset IDs
#   06. Raw Dataset Discovery/Download
#   07. Raw Dataset Loading
#   08. Structural Validation
#   09. Target Column Resolution
#   10. Missing-Value Analysis
#   11. Duplicate Analysis
#   12. Constant/Zero-Variance Analysis
#   13. Identifier Candidate Detection
#   14. Numeric/Categorical Inventory
#   15. Raw Dataset Statistical Summary
#   16. Raw File SHA-256 Fingerprinting
#   17. Dataset Validation Engine
#   18. Validation Summary
#   19. Save Feature Inventories
#   20. Save Missingness Reports
#   21. Save Structural Reports
#   22. Build Validated Dataset Registry
#   23. Save Fingerprint Manifest
#   24. Provenance Manifest
#   25. Final Integrity Verification
#   26. Completion Summary
#
# IMPORTANT:
#   This notebook does NOT:
#       - impute missing values
#       - encode categorical variables
#       - scale numerical variables
#       - remove outliers
#       - split datasets
#       - train models
#       - generate synthetic data
#
# It establishes the trusted raw-data boundary for all downstream experiments.
# ==============================================================================

print("=" * 100)
print("NOTEBOOK 01 — RAW DATASET LOADING & VALIDATION")
print("=" * 100)

print("\nResearch Objective:")
print("Establish a reproducible, validated, fingerprinted raw-data foundation")
print("for all downstream SPP-GAN experiments.")

print("\nRaw-data principle:")
print("RAW DATA MUST NOT BE MODIFIED IN THIS NOTEBOOK.")

print("\nNotebook 01 initialization complete.")

NOTEBOOK 01 — RAW DATASET LOADING & VALIDATION

Research Objective:
Establish a reproducible, validated, fingerprinted raw-data foundation
for all downstream SPP-GAN experiments.

Raw-data principle:
RAW DATA MUST NOT BE MODIFIED IN THIS NOTEBOOK.

Notebook 01 initialization complete.


In [2]:
# ==============================================================================
# 2. IMPORT LIBRARIES
# ==============================================================================

import os
import sys
import json
import hashlib
import shutil
import warnings
import platform
import subprocess

from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

print("=" * 100)
print("2. IMPORT LIBRARIES")
print("=" * 100)

print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.platform()}")
print(f"pandas      : {pd.__version__}")
print(f"numpy       : {np.__version__}")

print("\n✓ Core libraries imported successfully.")

2. IMPORT LIBRARIES
Python      : 3.13.15
Platform    : Linux-6.6.122+-x86_64-with-glibc2.35
pandas      : 2.2.3
numpy       : 2.1.3

✓ Core libraries imported successfully.


In [3]:
# ==============================================================================
# 3. LOAD NOTEBOOK 00 CONFIGURATION
# ==============================================================================
#
# Purpose:
#   Mount Google Drive, load/recover the canonical configuration established
#   in Notebook 00, and verify that the SPP-GAN project root is accessible.
#
# Design:
#   - Google Colab compatible
#   - Safe to rerun
#   - Does not delete or overwrite existing files
#   - Prefers configuration from Notebook 00
#   - Provides a controlled fallback when Notebook 00 variables are unavailable
# ==============================================================================

print("=" * 100)
print("3. LOAD NOTEBOOK 00 CONFIGURATION")
print("=" * 100)

# ==============================================================================
# 3.1 GOOGLE COLAB / GOOGLE DRIVE SETUP
# ==============================================================================

print("\n[3.1] Google Drive setup")

IN_COLAB = (
    "google.colab" in sys.modules
)

print(
    f"Google Colab environment : "
    f"{'YES' if IN_COLAB else 'NO'}"
)

if IN_COLAB:

    try:

        from google.colab import drive

        DRIVE_MOUNT_POINT = Path(
            "/content/drive"
        )

        DRIVE_MYDRIVE = (
            DRIVE_MOUNT_POINT / "MyDrive"
        )

        # ----------------------------------------------------------------------
        # Mount only when MyDrive is not already accessible
        # ----------------------------------------------------------------------

        if DRIVE_MYDRIVE.exists():

            print(
                "✓ Google Drive is already mounted."
            )

        else:

            print(
                "Mounting Google Drive..."
            )

            drive.mount(
                str(DRIVE_MOUNT_POINT),
                force_remount=False
            )

            print(
                "✓ Google Drive mounted successfully."
            )

        # ----------------------------------------------------------------------
        # Verify MyDrive
        # ----------------------------------------------------------------------

        if not DRIVE_MYDRIVE.exists():

            raise RuntimeError(
                "Google Drive mount completed, but "
                "/content/drive/MyDrive could not be verified."
            )

        print(
            f"✓ MyDrive accessible:"
            f"\n  {DRIVE_MYDRIVE}"
        )

    except Exception as exc:

        raise RuntimeError(
            "Google Drive initialization failed.\n"
            f"Error: {exc}"
        ) from exc

else:

    print(
        "⚠ Not running inside Google Colab."
    )

    DRIVE_MOUNT_POINT = Path(
        "/content/drive"
    )

    DRIVE_MYDRIVE = (
        DRIVE_MOUNT_POINT / "MyDrive"
    )

# ==============================================================================
# 3.2 EXPECTED NOTEBOOK 00 OBJECTS
# ==============================================================================

print("\n[3.2] Checking Notebook 00 configuration objects")

EXPECTED_CONFIG_OBJECTS = [
    "PROJECT_ROOT",
    "DIRECTORIES",
]

available_objects = {
    name: name in globals()
    for name in EXPECTED_CONFIG_OBJECTS
}

print("\nConfiguration object availability:")

for name, available in available_objects.items():

    print(
        f"  {'✓ AVAILABLE' if available else '⚠ NOT FOUND':<14}"
        f"{name}"
    )

# ==============================================================================
# 3.3 RESOLVE PROJECT ROOT
# ==============================================================================

print("\n[3.3] Resolving PROJECT_ROOT")

if "PROJECT_ROOT" in globals():

    PROJECT_ROOT = Path(
        PROJECT_ROOT
    )

    print(
        "✓ PROJECT_ROOT recovered from Notebook 00."
    )

else:

    # --------------------------------------------------------------------------
    # Controlled Colab fallback
    # --------------------------------------------------------------------------

    PROJECT_ROOT = (
        DRIVE_MYDRIVE
        / "SPP_GAN_Research"
    )

    print(
        "⚠ PROJECT_ROOT was not found in the active namespace."
    )

    print(
        "  Applying controlled Google Drive fallback:"
    )

    print(
        f"  {PROJECT_ROOT}"
    )

# ==============================================================================
# 3.4 VERIFY PROJECT ROOT LOCATION
# ==============================================================================

print("\n[3.4] Verifying project root")

PROJECT_ROOT = Path(
    PROJECT_ROOT
).expanduser()

print(
    f"Canonical project root:"
    f"\n  {PROJECT_ROOT}"
)

# Check whether project root is actually under MyDrive
if IN_COLAB:

    try:

        PROJECT_ROOT.relative_to(
            DRIVE_MYDRIVE
        )

        print(
            "✓ Project root is located inside Google Drive."
        )

    except ValueError:

        print(
            "⚠ WARNING: PROJECT_ROOT is not located "
            "inside /content/drive/MyDrive."
        )

# ==============================================================================
# 3.5 RECOVER DIRECTORY REGISTRY
# ==============================================================================

print("\n[3.5] Loading DIRECTORY registry")

if "DIRECTORIES" in globals():

    DIRECTORIES = {
        key: Path(value)
        for key, value in DIRECTORIES.items()
    }

    print(
        "✓ DIRECTORIES recovered from Notebook 00."
    )

else:

    print(
        "⚠ DIRECTORIES was not found in the active namespace."
    )

    print(
        "  Reconstructing the canonical SPP-GAN directory registry."
    )

    DIRECTORIES = {

        # ----------------------------------------------------------------------
        # Data
        # ----------------------------------------------------------------------

        "data":
            PROJECT_ROOT / "data",

        "raw_data":
            PROJECT_ROOT / "data" / "raw",

        "processed_data":
            PROJECT_ROOT / "data" / "processed",

        # ----------------------------------------------------------------------
        # Models
        # ----------------------------------------------------------------------

        "models":
            PROJECT_ROOT / "models",

        "tvae_models":
            PROJECT_ROOT / "models" / "tvae",

        "ctgan_models":
            PROJECT_ROOT / "models" / "ctgan",

        "dp_ctgan_models":
            PROJECT_ROOT / "models" / "dp_ctgan",

        "spp_gan_models":
            PROJECT_ROOT / "models" / "spp_gan",

        # ----------------------------------------------------------------------
        # Synthetic data
        # ----------------------------------------------------------------------

        "synthetic_data":
            PROJECT_ROOT / "synthetic_data",

        "tvae_synthetic":
            PROJECT_ROOT / "synthetic_data" / "tvae",

        "ctgan_synthetic":
            PROJECT_ROOT / "synthetic_data" / "ctgan",

        "dp_ctgan_synthetic":
            PROJECT_ROOT / "synthetic_data" / "dp_ctgan",

        "spp_gan_synthetic":
            PROJECT_ROOT / "synthetic_data" / "spp_gan",

        # ----------------------------------------------------------------------
        # Results
        # ----------------------------------------------------------------------

        "results":
            PROJECT_ROOT / "results",

        "statistical_results":
            PROJECT_ROOT / "results" / "statistical",

        "ml_results":
            PROJECT_ROOT / "results" / "machine_learning",

        "privacy_results":
            PROJECT_ROOT / "results" / "privacy",

        "utility_results":
            PROJECT_ROOT / "results" / "utility",

        "comparative_results":
            PROJECT_ROOT / "results" / "comparative",

        "raw_validation_results":
            PROJECT_ROOT / "results" / "raw_validation",

        # ----------------------------------------------------------------------
        # Reproducibility / logs
        # ----------------------------------------------------------------------

        "logs":
            PROJECT_ROOT / "logs",

        "checkpoints":
            PROJECT_ROOT / "checkpoints",

        "manifests":
            PROJECT_ROOT / "manifests",

        # ----------------------------------------------------------------------
        # Paper artifacts
        # ----------------------------------------------------------------------

        "figures":
            PROJECT_ROOT / "paper" / "figures",

        "tables":
            PROJECT_ROOT / "paper" / "tables",

        # ----------------------------------------------------------------------
        # Configuration
        # ----------------------------------------------------------------------

        "config":
            PROJECT_ROOT / "config",
    }

    print(
        "✓ Canonical DIRECTORY registry reconstructed."
    )

# ==============================================================================
# 3.6 ENSURE ALL VALUES ARE PATH OBJECTS
# ==============================================================================

DIRECTORIES = {
    key: Path(value)
    for key, value in DIRECTORIES.items()
}

# ==============================================================================
# 3.7 DISPLAY IMPORTANT PATHS
# ==============================================================================

print("\n[3.6] Canonical project paths")

print(
    f"  PROJECT_ROOT : {PROJECT_ROOT}"
)

print(
    f"  RAW_DATA     : {DIRECTORIES['raw_data']}"
)

print(
    f"  RESULTS      : {DIRECTORIES['results']}"
)

print(
    f"  MANIFESTS    : {DIRECTORIES['manifests']}"
)

# ==============================================================================
# 3.8 CONFIGURATION INTEGRITY CHECK
# ==============================================================================

print("\n[3.7] Configuration integrity check")

CONFIGURATION_CHECKS = {

    "project_root_is_path":
        isinstance(
            PROJECT_ROOT,
            Path
        ),

    "project_root_parent_accessible":
        PROJECT_ROOT.parent.exists(),

    "directories_is_dictionary":
        isinstance(
            DIRECTORIES,
            dict
        ),

    "raw_data_registered":
        "raw_data" in DIRECTORIES,

    "results_registered":
        "results" in DIRECTORIES,

    "manifests_registered":
        "manifests" in DIRECTORIES,

}

for check_name, passed in CONFIGURATION_CHECKS.items():

    print(
        f"  {'PASS' if passed else 'FAIL':<6} "
        f"{check_name}"
    )

CONFIGURATION_PASS = all(
    CONFIGURATION_CHECKS.values()
)

if not CONFIGURATION_PASS:

    failed_checks = [
        name
        for name, passed
        in CONFIGURATION_CHECKS.items()
        if not passed
    ]

    raise RuntimeError(
        "Notebook 00 configuration verification failed:\n"
        + "\n".join(
            f"  - {name}"
            for name in failed_checks
        )
    )

# ==============================================================================
# 3.9 FINAL STATUS
# ==============================================================================

print("\n" + "=" * 100)
print("NOTEBOOK 00 CONFIGURATION LOADED SUCCESSFULLY")
print("=" * 100)

print(
    f"Google Drive          : "
    f"{'MOUNTED / ACCESSIBLE' if IN_COLAB else 'NOT APPLICABLE'}"
)

print(
    f"Project Root          : "
    f"{PROJECT_ROOT}"
)

print(
    f"Registered Directories : "
    f"{len(DIRECTORIES)}"
)

print(
    f"Configuration Status  : PASS"
)

print("=" * 100)

3. LOAD NOTEBOOK 00 CONFIGURATION

[3.1] Google Drive setup
Google Colab environment : YES
Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted successfully.
✓ MyDrive accessible:
  /content/drive/MyDrive

[3.2] Checking Notebook 00 configuration objects

Configuration object availability:
  ⚠ NOT FOUND   PROJECT_ROOT
  ⚠ NOT FOUND   DIRECTORIES

[3.3] Resolving PROJECT_ROOT
⚠ PROJECT_ROOT was not found in the active namespace.
  Applying controlled Google Drive fallback:
  /content/drive/MyDrive/SPP_GAN_Research

[3.4] Verifying project root
Canonical project root:
  /content/drive/MyDrive/SPP_GAN_Research
✓ Project root is located inside Google Drive.

[3.5] Loading DIRECTORY registry
⚠ DIRECTORIES was not found in the active namespace.
  Reconstructing the canonical SPP-GAN directory registry.
✓ Canonical DIRECTORY registry reconstructed.

[3.6] Canonical project paths
  PROJECT_ROOT : /content/drive/MyDrive/SPP_GAN_Research
  RAW_DATA     : /content/drive/MyDri

In [4]:
# ==============================================================================
# 4. VERIFY PROJECT STRUCTURE
# ==============================================================================

print("=" * 100)
print("4. VERIFY PROJECT STRUCTURE")
print("=" * 100)

required_directory_keys = [
    "data",
    "raw_data",
    "processed_data",
    "models",
    "results",
    "logs",
    "manifests",
    "config",
]

missing_directories = []

for key in required_directory_keys:

    path = DIRECTORIES.get(key)

    if path is None:
        missing_directories.append((key, "NOT REGISTERED"))
        continue

    if not path.exists():
        missing_directories.append((key, str(path)))

        # Safe creation of expected directories
        path.mkdir(parents=True, exist_ok=True)

        print(f"  CREATED : {key:<20} → {path}")

    else:
        print(f"  ✓ EXISTS : {key:<20} → {path}")

# ------------------------------------------------------------------------------
# Verify project root
# ------------------------------------------------------------------------------

if not PROJECT_ROOT.exists():
    raise RuntimeError(
        f"PROJECT_ROOT does not exist: {PROJECT_ROOT}"
    )

print("\nProject structure verification:")
print(f"  Project root : PASS")
print(f"  Raw data dir : {DIRECTORIES['raw_data']}")
print(f"  Results dir  : {DIRECTORIES['results']}")

print("\n✓ Project structure verified.")

4. VERIFY PROJECT STRUCTURE
  ✓ EXISTS : data                 → /content/drive/MyDrive/SPP_GAN_Research/data
  ✓ EXISTS : raw_data             → /content/drive/MyDrive/SPP_GAN_Research/data/raw
  ✓ EXISTS : processed_data       → /content/drive/MyDrive/SPP_GAN_Research/data/processed
  ✓ EXISTS : models               → /content/drive/MyDrive/SPP_GAN_Research/models
  ✓ EXISTS : results              → /content/drive/MyDrive/SPP_GAN_Research/results
  ✓ EXISTS : logs                 → /content/drive/MyDrive/SPP_GAN_Research/logs
  ✓ EXISTS : manifests            → /content/drive/MyDrive/SPP_GAN_Research/manifests
  ✓ EXISTS : config               → /content/drive/MyDrive/SPP_GAN_Research/config

Project structure verification:
  Project root : PASS
  Raw data dir : /content/drive/MyDrive/SPP_GAN_Research/data/raw
  Results dir  : /content/drive/MyDrive/SPP_GAN_Research/results

✓ Project structure verified.


In [5]:
# ==============================================================================
# 5. RESOLVE DATASET IDS
# ==============================================================================

print("=" * 100)
print("5. RESOLVE DATASET IDS")
print("=" * 100)

# ------------------------------------------------------------------------------
# Canonical SPP-GAN dataset registry
#
# IMPORTANT:
#   File paths are resolved later.
#   Dataset IDs themselves must remain stable throughout the research project.
# ------------------------------------------------------------------------------

DEFAULT_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

# ------------------------------------------------------------------------------
# Prefer Notebook 00 configuration if available
# ------------------------------------------------------------------------------

if "DATASET_IDS" in globals():

    DATASET_IDS = list(DATASET_IDS)

elif "DATASETS" in globals():

    if isinstance(DATASETS, dict):
        DATASET_IDS = list(DATASETS.keys())

    elif isinstance(DATASETS, (list, tuple)):
        DATASET_IDS = list(DATASETS)

    else:
        DATASET_IDS = DEFAULT_DATASET_IDS

else:
    DATASET_IDS = DEFAULT_DATASET_IDS

# ------------------------------------------------------------------------------
# Normalize
# ------------------------------------------------------------------------------

DATASET_IDS = [
    str(dataset_id).strip()
    for dataset_id in DATASET_IDS
    if str(dataset_id).strip()
]

DATASET_IDS = list(dict.fromkeys(DATASET_IDS))

# ------------------------------------------------------------------------------
# Integrity
# ------------------------------------------------------------------------------

if not DATASET_IDS:
    raise RuntimeError("No dataset IDs were resolved.")

print("\nResolved dataset IDs:")

for idx, dataset_id in enumerate(DATASET_IDS, start=1):
    print(f"  {idx}. {dataset_id}")

print(f"\nTotal datasets: {len(DATASET_IDS)}")

# ------------------------------------------------------------------------------
# Research consistency check
# ------------------------------------------------------------------------------

EXPECTED_DATASET_COUNT = 3

if len(DATASET_IDS) != EXPECTED_DATASET_COUNT:

    print(
        f"\n⚠ WARNING: Expected {EXPECTED_DATASET_COUNT} datasets "
        f"for the current SPP-GAN experimental design, "
        f"but resolved {len(DATASET_IDS)}."
    )

else:
    print("\n✓ Expected three-dataset experimental design confirmed.")

5. RESOLVE DATASET IDS

Resolved dataset IDs:
  1. adult_income
  2. bank_marketing
  3. diabetes_130us

Total datasets: 3

✓ Expected three-dataset experimental design confirmed.


In [6]:
# ==============================================================================
# 6. RAW DATASET DISCOVERY / DOWNLOAD
# ==============================================================================

print("=" * 100)
print("6. RAW DATASET DISCOVERY / DOWNLOAD")
print("=" * 100)

RAW_DATA_DIR = Path(DIRECTORIES["raw_data"])

SUPPORTED_EXTENSIONS = {
    ".csv",
    ".csv.gz",
    ".parquet",
    ".json",
    ".xlsx",
    ".xls",
    ".feather",
}

# ------------------------------------------------------------------------------
# Optional configured source registry
# ------------------------------------------------------------------------------

if "DATASET_REGISTRY" in globals():

    CONFIG_DATASET_REGISTRY = DATASET_REGISTRY

elif "DATASETS" in globals():

    CONFIG_DATASET_REGISTRY = DATASETS

else:

    CONFIG_DATASET_REGISTRY = {
        dataset_id: {}
        for dataset_id in DATASET_IDS
    }

# ------------------------------------------------------------------------------
# Discovery helper
# ------------------------------------------------------------------------------

def discover_dataset_files(dataset_id, search_root):
    """
    Discover candidate raw files associated with a dataset ID.

    Matching is intentionally conservative:
      - exact stem match
      - normalized name match
      - dataset ID contained in filename
    """

    dataset_id_norm = (
        str(dataset_id)
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )

    candidates = []

    for path in search_root.rglob("*"):

        if not path.is_file():
            continue

        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            continue

        filename_norm = (
            path.name
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        stem_norm = (
            path.stem
            .lower()
            .replace("-", "_")
            .replace(" ", "_")
        )

        if (
            dataset_id_norm == stem_norm
            or dataset_id_norm in filename_norm
        ):
            candidates.append(path)

    return sorted(
        candidates,
        key=lambda p: (
            len(p.name),
            str(p)
        )
    )

# ------------------------------------------------------------------------------
# Discover
# ------------------------------------------------------------------------------

RAW_DATASET_DISCOVERY = {}

for dataset_id in DATASET_IDS:

    candidates = discover_dataset_files(
        dataset_id=dataset_id,
        search_root=RAW_DATA_DIR
    )

    RAW_DATASET_DISCOVERY[dataset_id] = candidates

    print(f"\nDataset: {dataset_id}")

    if candidates:

        for candidate in candidates:
            print(f"  ✓ {candidate}")

    else:
        print("  ⚠ No local raw file discovered.")

print("\n✓ Raw dataset discovery completed.")

6. RAW DATASET DISCOVERY / DOWNLOAD

Dataset: adult_income
  ✓ /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv

Dataset: bank_marketing
  ✓ /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv

Dataset: diabetes_130us
  ✓ /content/drive/MyDrive/SPP_GAN_Research/data/raw/diabetes_130us.csv

✓ Raw dataset discovery completed.


In [7]:
# ==================================================================================================
# 7. RAW DATASET LOADING
# ==================================================================================================

print("=" * 100)
print("7. RAW DATASET LOADING")
print("=" * 100)

import pandas as pd
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 7.1 Dataset-specific raw-file parsing configuration
# --------------------------------------------------------------------------------------------------

RAW_READ_CONFIG = {
    "adult_income": {
        "sep": ",",
        "encoding": "utf-8",
        "low_memory": False,
    },

    "bank_marketing": {
        "sep": ";",
        "encoding": "utf-8",
        "low_memory": False,
    },

    "diabetes_130us": {
        "sep": ",",
        "encoding": "utf-8",
        "low_memory": False,
    },
}


# --------------------------------------------------------------------------------------------------
# 7.2 Expected raw dataset structure
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_STRUCTURE = {
    "adult_income": {
        "min_rows": 1,
        "min_columns": 15,
    },

    "bank_marketing": {
        "min_rows": 1,
        "min_columns": 17,
    },

    "diabetes_130us": {
        "min_rows": 1,
        "min_columns": 50,
    },
}


# --------------------------------------------------------------------------------------------------
# 7.3 Resolve raw-file mapping created by Section 6
# --------------------------------------------------------------------------------------------------

RAW_FILE_MAPPING = None
RAW_FILE_MAPPING_SOURCE = None


# Preferred variable names, checked in order.
_MAPPING_CANDIDATES = [
    "RAW_DATASET_PATHS",
    "RAW_DATASET_FILES",
    "DATASET_FILES",
    "DATASET_PATHS",
    "RAW_FILES",
]


for _candidate_name in _MAPPING_CANDIDATES:

    if _candidate_name in globals():

        _candidate_value = globals()[_candidate_name]

        if isinstance(_candidate_value, dict):

            # Check whether the dictionary contains at least
            # one of our registered dataset IDs.
            _matched_ids = [
                dataset_id
                for dataset_id in DATASET_IDS
                if dataset_id in _candidate_value
            ]

            if _matched_ids:

                RAW_FILE_MAPPING = _candidate_value
                RAW_FILE_MAPPING_SOURCE = _candidate_name
                break


# --------------------------------------------------------------------------------------------------
# 7.4 If Section 6 did not expose a mapping, recover files directly
# --------------------------------------------------------------------------------------------------

if RAW_FILE_MAPPING is None:

    print()
    print("⚠ No raw-file mapping variable was found from Section 6.")
    print("  Attempting deterministic recovery from the raw-data directory...")
    print()

    if "DIRECTORIES" not in globals():
        raise RuntimeError(
            "DIRECTORIES is not available.\n"
            "Execute Notebook 00 / Section 3 before Section 7."
        )

    if "DATASET_IDS" not in globals():
        raise RuntimeError(
            "DATASET_IDS is not available.\n"
            "Execute Section 5 before Section 7."
        )

    RAW_DATA_DIR = Path(DIRECTORIES["raw_data"])

    if not RAW_DATA_DIR.exists():
        raise FileNotFoundError(
            f"Raw data directory does not exist:\n{RAW_DATA_DIR}"
        )

    SUPPORTED_EXTENSIONS = {
        ".csv",
        ".csv.gz",
        ".parquet",
        ".json",
        ".xlsx",
        ".xls",
        ".feather",
    }

    def _full_suffix(path):
        return "".join(path.suffixes).lower()

    raw_candidates = sorted(
        [
            path
            for path in RAW_DATA_DIR.rglob("*")
            if path.is_file()
            and (
                path.suffix.lower() in SUPPORTED_EXTENSIONS
                or _full_suffix(path) in SUPPORTED_EXTENSIONS
            )
        ],
        key=lambda x: str(x).lower()
    )

    recovered_mapping = {}

    for dataset_id in DATASET_IDS:

        matches = [
            path
            for path in raw_candidates
            if dataset_id.lower() in path.name.lower()
        ]

        if len(matches) == 1:

            recovered_mapping[dataset_id] = matches[0]

        elif len(matches) == 0:

            raise FileNotFoundError(
                f"No raw file found for dataset '{dataset_id}'.\n"
                f"Raw data directory:\n{RAW_DATA_DIR}"
            )

        else:

            print(
                f"Multiple files found for '{dataset_id}':"
            )

            for match in matches:
                print(f"  - {match}")

            raise RuntimeError(
                f"Ambiguous raw-file selection for '{dataset_id}'. "
                f"Please specify the exact raw file in Notebook 00/Section 6."
            )

    RAW_FILE_MAPPING = recovered_mapping
    RAW_FILE_MAPPING_SOURCE = "DIRECTORY_RECOVERY"


# --------------------------------------------------------------------------------------------------
# 7.5 Normalize raw-file paths
# --------------------------------------------------------------------------------------------------

RAW_FILE_MAPPING = {
    dataset_id: Path(path)
    for dataset_id, path in RAW_FILE_MAPPING.items()
    if dataset_id in DATASET_IDS
}


# --------------------------------------------------------------------------------------------------
# 7.6 Verify all required datasets have raw files
# --------------------------------------------------------------------------------------------------

MISSING_RAW_FILES = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in RAW_FILE_MAPPING
]


if MISSING_RAW_FILES:

    raise RuntimeError(
        "Raw files are missing for:\n"
        + "\n".join(f"  - {dataset_id}" for dataset_id in MISSING_RAW_FILES)
    )


# --------------------------------------------------------------------------------------------------
# 7.7 Helper: determine file format
# --------------------------------------------------------------------------------------------------

def get_file_format(path):

    path = Path(path)

    suffixes = "".join(path.suffixes).lower()

    if suffixes.endswith(".csv.gz"):
        return "csv"

    if suffixes.endswith(".csv"):
        return "csv"

    if suffixes.endswith(".parquet"):
        return "parquet"

    if suffixes.endswith(".json"):
        return "json"

    if suffixes.endswith(".xlsx"):
        return "xlsx"

    if suffixes.endswith(".xls"):
        return "xls"

    if suffixes.endswith(".feather"):
        return "feather"

    return None


# --------------------------------------------------------------------------------------------------
# 7.8 Helper: load one raw dataset
# --------------------------------------------------------------------------------------------------

def load_raw_dataset(dataset_id, path):

    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(
            f"Raw dataset file does not exist:\n{path}"
        )

    file_format = get_file_format(path)

    if file_format is None:
        raise ValueError(
            f"Unsupported raw dataset format:\n{path}"
        )

    config = RAW_READ_CONFIG.get(
        dataset_id,
        {
            "sep": ",",
            "encoding": "utf-8",
            "low_memory": False,
        }
    )

    if file_format == "csv":

        df = pd.read_csv(
            path,
            sep=config.get("sep", ","),
            encoding=config.get("encoding", "utf-8"),
            low_memory=config.get("low_memory", False),
        )

    elif file_format == "parquet":

        df = pd.read_parquet(path)

    elif file_format == "json":

        df = pd.read_json(
            path,
            encoding=config.get("encoding", "utf-8"),
        )

    elif file_format in {"xlsx", "xls"}:

        df = pd.read_excel(path)

    elif file_format == "feather":

        df = pd.read_feather(path)

    else:

        raise RuntimeError(
            f"Unhandled file format: {file_format}"
        )

    if not isinstance(df, pd.DataFrame):

        raise TypeError(
            f"Loaded object for '{dataset_id}' is not a pandas DataFrame."
        )

    return df


# --------------------------------------------------------------------------------------------------
# 7.9 Reset raw dataset containers
# --------------------------------------------------------------------------------------------------

RAW_DATASETS = {}

RAW_DATASET_PATHS = {}

RAW_DATASET_LOAD_STATUS = {}

RAW_DATASET_LOAD_ERRORS = []


# --------------------------------------------------------------------------------------------------
# 7.10 Load datasets
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    try:

        raw_path = RAW_FILE_MAPPING[dataset_id]

        print(f"Raw file : {raw_path}")

        # ------------------------------------------------------------------------------------------
        # Verify file
        # ------------------------------------------------------------------------------------------

        if not raw_path.exists():

            raise FileNotFoundError(
                f"Raw dataset file does not exist:\n{raw_path}"
            )

        file_format = get_file_format(raw_path)

        print(f"Format   : {file_format}")

        # ------------------------------------------------------------------------------------------
        # Show parser settings
        # ------------------------------------------------------------------------------------------

        parser_config = RAW_READ_CONFIG.get(
            dataset_id,
            {
                "sep": ",",
                "encoding": "utf-8",
                "low_memory": False,
            }
        )

        if file_format == "csv":

            print(
                f"Delimiter: {repr(parser_config.get('sep', ','))}"
            )

            print(
                f"Encoding : {parser_config.get('encoding', 'utf-8')}"
            )

        # ------------------------------------------------------------------------------------------
        # Load
        # ------------------------------------------------------------------------------------------

        df = load_raw_dataset(
            dataset_id=dataset_id,
            path=raw_path,
        )

        # ------------------------------------------------------------------------------------------
        # Basic validation
        # ------------------------------------------------------------------------------------------

        if df.empty:

            raise ValueError(
                f"Dataset '{dataset_id}' loaded but contains zero rows."
            )

        if df.shape[1] == 0:

            raise ValueError(
                f"Dataset '{dataset_id}' loaded but contains zero columns."
            )

        # ------------------------------------------------------------------------------------------
        # Expected structural validation
        # ------------------------------------------------------------------------------------------

        expected = EXPECTED_DATASET_STRUCTURE.get(
            dataset_id,
            {}
        )

        min_rows = expected.get("min_rows", 1)
        min_columns = expected.get("min_columns", 1)

        if df.shape[0] < min_rows:

            raise ValueError(
                f"Dataset '{dataset_id}' has too few rows.\n"
                f"Loaded rows : {df.shape[0]:,}\n"
                f"Minimum     : {min_rows:,}"
            )

        if df.shape[1] < min_columns:

            raise ValueError(
                f"Dataset '{dataset_id}' appears to have been "
                f"parsed incorrectly.\n"
                f"Loaded columns : {df.shape[1]:,}\n"
                f"Expected >=    : {min_columns:,}\n"
                f"Delimiter      : {repr(parser_config.get('sep', ','))}\n"
                f"File           : {raw_path}"
            )

        # ------------------------------------------------------------------------------------------
        # Store
        # ------------------------------------------------------------------------------------------

        RAW_DATASETS[dataset_id] = df

        RAW_DATASET_PATHS[dataset_id] = raw_path

        RAW_DATASET_LOAD_STATUS[dataset_id] = "PASS"

        # ------------------------------------------------------------------------------------------
        # Report
        # ------------------------------------------------------------------------------------------

        memory_mb = (
            df.memory_usage(deep=True).sum()
            / (1024 ** 2)
        )

        print()
        print("✓ Dataset loaded successfully")

        print(
            f"  Rows    : {df.shape[0]:,}"
        )

        print(
            f"  Columns : {df.shape[1]:,}"
        )

        print(
            f"  Memory  : {memory_mb:.2f} MB"
        )

        print()
        print("  Columns:")

        for position, column in enumerate(df.columns, start=1):

            print(
                f"    {position:>2}. {column}"
            )

    except Exception as exc:

        RAW_DATASET_LOAD_STATUS[dataset_id] = "FAIL"

        RAW_DATASET_LOAD_ERRORS.append(
            {
                "dataset_id": dataset_id,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

        print()
        print(f"✗ FAILED: {dataset_id}")
        print(
            f"  {type(exc).__name__}: {exc}"
        )


# --------------------------------------------------------------------------------------------------
# 7.11 Loading summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("RAW DATASET LOADING SUMMARY")
print("=" * 100)

print()
print(
    f"Raw-file mapping source: {RAW_FILE_MAPPING_SOURCE}"
)

print()

for dataset_id in DATASET_IDS:

    status = RAW_DATASET_LOAD_STATUS.get(
        dataset_id,
        "NOT_ATTEMPTED"
    )

    if status == "PASS":

        df = RAW_DATASETS[dataset_id]

        print(
            f"✓ {dataset_id:<20} "
            f"{df.shape[0]:>10,} rows × "
            f"{df.shape[1]:>3} columns"
        )

    else:

        print(
            f"✗ {dataset_id:<20} "
            f"STATUS = {status}"
        )


# --------------------------------------------------------------------------------------------------
# 7.12 Hard integrity check
# --------------------------------------------------------------------------------------------------

FAILED_DATASETS = [
    dataset_id
    for dataset_id in DATASET_IDS
    if RAW_DATASET_LOAD_STATUS.get(dataset_id) != "PASS"
]


if FAILED_DATASETS:

    print()
    print("FAILED DATASETS:")

    for dataset_id in FAILED_DATASETS:
        print(f"  - {dataset_id}")

    raise RuntimeError(
        "Raw dataset loading failed."
    )


# --------------------------------------------------------------------------------------------------
# 7.13 Final success
# --------------------------------------------------------------------------------------------------

print()
print("✓ All raw datasets loaded successfully.")
print("✓ Dataset-specific parsing rules applied.")
print("✓ Bank Marketing semicolon delimiter handled correctly.")
print("✓ Raw datasets remain unmodified.")
print("✓ Section 7 completed successfully.")

7. RAW DATASET LOADING

⚠ No raw-file mapping variable was found from Section 6.
  Attempting deterministic recovery from the raw-data directory...


----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Raw file : /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv
Format   : csv
Delimiter: ','
Encoding : utf-8

✓ Dataset loaded successfully
  Rows    : 48,842
  Columns : 15
  Memory  : 26.36 MB

  Columns:
     1. age
     2. workclass
     3. fnlwgt
     4. education
     5. education_num
     6. marital_status
     7. occupation
     8. relationship
     9. race
    10. sex
    11. capital_gain
    12. capital_loss
    13. hours_per_week
    14. native_country
    15. income

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
--

In [8]:
# ==============================================================================
# 8. STRUCTURAL VALIDATION
# ==============================================================================

print("=" * 100)
print("8. STRUCTURAL VALIDATION")
print("=" * 100)

STRUCTURAL_REPORTS = []

for dataset_id, df in RAW_DATASETS.items():

    columns = list(df.columns)

    duplicate_column_names = (
        pd.Index(columns).duplicated().sum()
    )

    unnamed_columns = [
        str(col)
        for col in columns
        if str(col).lower().startswith("unnamed:")
    ]

    null_column_names = [
        str(col)
        for col in columns
        if pd.isna(col)
    ]

    report = {
        "dataset_id": dataset_id,
        "n_rows": int(df.shape[0]),
        "n_columns": int(df.shape[1]),
        "memory_usage_bytes": int(
            df.memory_usage(deep=True).sum()
        ),
        "duplicate_column_names": int(
            duplicate_column_names
        ),
        "unnamed_columns_count": len(
            unnamed_columns
        ),
        "null_column_names_count": len(
            null_column_names
        ),
        "index_unique": bool(df.index.is_unique),
        "empty_dataset": bool(
            df.shape[0] == 0 or df.shape[1] == 0
        ),
    }

    STRUCTURAL_REPORTS.append(report)

    print(f"\nDataset: {dataset_id}")
    print(f"  Rows                  : {report['n_rows']:,}")
    print(f"  Columns               : {report['n_columns']:,}")
    print(
        f"  Duplicate column names: "
        f"{report['duplicate_column_names']}"
    )
    print(
        f"  Unnamed columns       : "
        f"{report['unnamed_columns_count']}"
    )
    print(
        f"  Empty dataset         : "
        f"{report['empty_dataset']}"
    )

STRUCTURAL_REPORT_DF = pd.DataFrame(
    STRUCTURAL_REPORTS
)

print("\n✓ Structural validation completed.")

8. STRUCTURAL VALIDATION

Dataset: adult_income
  Rows                  : 48,842
  Columns               : 15
  Duplicate column names: 0
  Unnamed columns       : 0
  Empty dataset         : False

Dataset: bank_marketing
  Rows                  : 45,211
  Columns               : 17
  Duplicate column names: 0
  Unnamed columns       : 0
  Empty dataset         : False

Dataset: diabetes_130us
  Rows                  : 101,766
  Columns               : 50
  Duplicate column names: 0
  Unnamed columns       : 0
  Empty dataset         : False

✓ Structural validation completed.


In [9]:
# ==================================================================================================
# 9. TARGET COLUMN RESOLUTION
# ==================================================================================================

print("=" * 100)
print("9. TARGET COLUMN RESOLUTION")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 9.1 Explicit research target configuration
# --------------------------------------------------------------------------------------------------
#
# These targets correspond to the outcome variables in the three raw datasets.
#
# IMPORTANT:
# - Target columns are NOT inferred from column names.
# - Target columns are NOT removed here.
# - Target columns remain part of the raw dataset.
# - This section only identifies and validates the target.
#
# --------------------------------------------------------------------------------------------------

RESEARCH_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}


# --------------------------------------------------------------------------------------------------
# 9.2 Verify dataset registry
# --------------------------------------------------------------------------------------------------

if "DATASET_IDS" not in globals():
    raise RuntimeError(
        "DATASET_IDS is not available.\n"
        "Execute Section 5 before Section 9."
    )

if "RAW_DATASETS" not in globals():
    raise RuntimeError(
        "RAW_DATASETS is not available.\n"
        "Execute Section 7 before Section 9."
    )


# --------------------------------------------------------------------------------------------------
# 9.3 Verify target configuration coverage
# --------------------------------------------------------------------------------------------------

missing_target_configuration = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in RESEARCH_TARGET_COLUMNS
]

if missing_target_configuration:

    raise RuntimeError(
        "Target configuration is missing for:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_target_configuration
        )
    )


# --------------------------------------------------------------------------------------------------
# 9.4 Resolve and validate targets
# --------------------------------------------------------------------------------------------------

TARGET_COLUMNS = {}

TARGET_RESOLUTION_REPORT = []


for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    # ----------------------------------------------------------------------------------------------
    # Verify dataset is loaded
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in RAW_DATASETS:

        raise RuntimeError(
            f"Dataset '{dataset_id}' is not present in RAW_DATASETS."
        )

    df = RAW_DATASETS[dataset_id]

    # ----------------------------------------------------------------------------------------------
    # Configured target
    # ----------------------------------------------------------------------------------------------

    configured_target = RESEARCH_TARGET_COLUMNS[dataset_id]

    print(
        f"Configured target : {configured_target!r}"
    )

    # ----------------------------------------------------------------------------------------------
    # Exact column existence check
    # ----------------------------------------------------------------------------------------------

    if configured_target not in df.columns:

        raise ValueError(
            f"Configured target '{configured_target}' was not found "
            f"in dataset '{dataset_id}'.\n\n"
            f"Available columns:\n"
            + "\n".join(
                f"  - {column!r}"
                for column in df.columns
            )
        )

    # ----------------------------------------------------------------------------------------------
    # Target data inspection
    # ----------------------------------------------------------------------------------------------

    target_series = df[configured_target]

    target_dtype = str(target_series.dtype)

    target_missing_count = int(
        target_series.isna().sum()
    )

    target_missing_rate = (
        target_missing_count / len(df)
        if len(df) > 0
        else 0.0
    )

    target_unique_count = int(
        target_series.nunique(dropna=True)
    )

    # ----------------------------------------------------------------------------------------------
    # Store resolved target
    # ----------------------------------------------------------------------------------------------

    TARGET_COLUMNS[dataset_id] = configured_target

    # ----------------------------------------------------------------------------------------------
    # Build validation record
    # ----------------------------------------------------------------------------------------------

    TARGET_RESOLUTION_REPORT.append(
        {
            "dataset_id": dataset_id,
            "target_column": configured_target,
            "target_dtype": target_dtype,
            "target_unique_values": target_unique_count,
            "target_missing_count": target_missing_count,
            "target_missing_rate": target_missing_rate,
            "target_exists": True,
            "target_resolution_status": "PASS",
        }
    )

    # ----------------------------------------------------------------------------------------------
    # Report
    # ----------------------------------------------------------------------------------------------

    print(
        f"✓ Target resolved      : {configured_target!r}"
    )

    print(
        f"  Data type            : {target_dtype}"
    )

    print(
        f"  Unique values        : {target_unique_count:,}"
    )

    print(
        f"  Missing values       : {target_missing_count:,}"
    )

    print(
        f"  Missing rate         : {target_missing_rate:.6f}"
    )


# --------------------------------------------------------------------------------------------------
# 9.5 Convert resolution report to DataFrame
# --------------------------------------------------------------------------------------------------

TARGET_RESOLUTION_DF = pd.DataFrame(
    TARGET_RESOLUTION_REPORT
)


# --------------------------------------------------------------------------------------------------
# 9.6 Target configuration integrity check
# --------------------------------------------------------------------------------------------------

if len(TARGET_COLUMNS) != len(DATASET_IDS):

    raise RuntimeError(
        "Target resolution incomplete.\n"
        f"Expected datasets : {len(DATASET_IDS)}\n"
        f"Resolved datasets : {len(TARGET_COLUMNS)}"
    )


if not TARGET_RESOLUTION_DF["target_exists"].all():

    raise RuntimeError(
        "One or more target columns failed validation."
    )


# --------------------------------------------------------------------------------------------------
# 9.7 Final target mapping
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("RESOLVED TARGET COLUMNS")
print("=" * 100)

for dataset_id in DATASET_IDS:

    print(
        f"✓ {dataset_id:<20} → "
        f"{TARGET_COLUMNS[dataset_id]!r}"
    )


# --------------------------------------------------------------------------------------------------
# 9.8 Final status
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("TARGET RESOLUTION SUMMARY")
print("=" * 100)

print(
    f"Datasets evaluated : {len(DATASET_IDS)}"
)

print(
    f"Targets resolved   : {len(TARGET_COLUMNS)}"
)

print(
    f"Validation status  : PASS"
)

print()
print("✓ Target columns explicitly configured.")
print("✓ Target columns verified against raw dataset schemas.")
print("✓ No target column was removed or modified.")
print("✓ Target resolution completed successfully.")

9. TARGET COLUMN RESOLUTION

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
Configured target : 'income'
✓ Target resolved      : 'income'
  Data type            : object
  Unique values        : 2
  Missing values       : 0
  Missing rate         : 0.000000

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
Configured target : 'y'
✓ Target resolved      : 'y'
  Data type            : object
  Unique values        : 2
  Missing values       : 0
  Missing rate         : 0.000000

----------------------------------------------------------------------------------------------------
Dataset: diabetes_130us
--------------------------------------

In [10]:
# ==============================================================================
# 10. MISSING-VALUE ANALYSIS
# ==============================================================================

print("=" * 100)
print("10. MISSING-VALUE ANALYSIS")
print("=" * 100)

MISSINGNESS_REPORTS = []

for dataset_id, df in RAW_DATASETS.items():

    n_rows = len(df)

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        missing_rate = (
            missing_count / n_rows
            if n_rows > 0
            else np.nan
        )

        MISSINGNESS_REPORTS.append({
            "dataset_id": dataset_id,
            "column": str(column),
            "dtype": str(df[column].dtype),
            "n_rows": int(n_rows),
            "missing_count": missing_count,
            "missing_rate": float(missing_rate),
            "observed_count": int(
                n_rows - missing_count
            ),
            "is_complete": bool(
                missing_count == 0
            ),
        })

MISSINGNESS_REPORT_DF = pd.DataFrame(
    MISSINGNESS_REPORTS
)

print(
    "\nColumns with missing values:",
    int(
        (MISSINGNESS_REPORT_DF["missing_count"] > 0).sum()
    )
)

print(
    "Total missing cells:",
    int(
        MISSINGNESS_REPORT_DF["missing_count"].sum()
    )
)

print("\n✓ Missing-value analysis completed.")

10. MISSING-VALUE ANALYSIS

Columns with missing values: 5
Total missing cells: 187633

✓ Missing-value analysis completed.


In [11]:
# ==============================================================================
# 11. DUPLICATE ANALYSIS
# ==============================================================================

print("=" * 100)
print("11. DUPLICATE ANALYSIS")
print("=" * 100)

DUPLICATE_REPORTS = []

for dataset_id, df in RAW_DATASETS.items():

    duplicate_mask = df.duplicated(
        keep=False
    )

    duplicate_row_count = int(
        duplicate_mask.sum()
    )

    duplicated_excluding_first = int(
        df.duplicated(
            keep="first"
        ).sum()
    )

    unique_row_count = int(
        len(df.drop_duplicates())
    )

    duplicate_rate = (
        duplicated_excluding_first / len(df)
        if len(df) > 0
        else np.nan
    )

    DUPLICATE_REPORTS.append({
        "dataset_id": dataset_id,
        "n_rows": int(len(df)),
        "duplicate_rows_including_all_members":
            duplicate_row_count,
        "duplicate_rows_excluding_first":
            duplicated_excluding_first,
        "unique_rows":
            unique_row_count,
        "duplicate_rate":
            float(duplicate_rate),
    })

DUPLICATE_REPORT_DF = pd.DataFrame(
    DUPLICATE_REPORTS
)

for record in DUPLICATE_REPORTS:

    print(
        f"\n{record['dataset_id']}"
    )

    print(
        f"  Duplicate rows : "
        f"{record['duplicate_rows_excluding_first']:,}"
    )

    print(
        f"  Duplicate rate : "
        f"{record['duplicate_rate']:.6f}"
    )

print("\n✓ Duplicate analysis completed.")

11. DUPLICATE ANALYSIS

adult_income
  Duplicate rows : 52
  Duplicate rate : 0.001065

bank_marketing
  Duplicate rows : 0
  Duplicate rate : 0.000000

diabetes_130us
  Duplicate rows : 0
  Duplicate rate : 0.000000

✓ Duplicate analysis completed.


In [12]:
# ==============================================================================
# 12. CONSTANT / ZERO-VARIANCE ANALYSIS
# ==============================================================================

print("=" * 100)
print("12. CONSTANT / ZERO-VARIANCE ANALYSIS")
print("=" * 100)

CONSTANT_REPORTS = []

for dataset_id, df in RAW_DATASETS.items():

    for column in df.columns:

        series = df[column]

        n_unique = int(
            series.nunique(
                dropna=True
            )
        )

        is_constant = (
            n_unique <= 1
        )

        is_numeric = pd.api.types.is_numeric_dtype(
            series
        )

        variance = np.nan

        if is_numeric:

            try:
                variance = float(
                    series.var(
                        skipna=True
                    )
                )
            except Exception:
                variance = np.nan

        zero_variance = (
            bool(
                is_numeric
                and pd.notna(variance)
                and np.isclose(variance, 0.0)
            )
        )

        CONSTANT_REPORTS.append({
            "dataset_id": dataset_id,
            "column": str(column),
            "dtype": str(series.dtype),
            "n_unique": n_unique,
            "variance": variance,
            "is_constant": bool(is_constant),
            "is_zero_variance": zero_variance,
        })

CONSTANT_REPORT_DF = pd.DataFrame(
    CONSTANT_REPORTS
)

print(
    "\nConstant columns:",
    int(
        CONSTANT_REPORT_DF["is_constant"].sum()
    )
)

print(
    "Numeric zero-variance columns:",
    int(
        CONSTANT_REPORT_DF["is_zero_variance"].sum()
    )
)

print("\n✓ Constant/zero-variance analysis completed.")

12. CONSTANT / ZERO-VARIANCE ANALYSIS

Constant columns: 2
Numeric zero-variance columns: 0

✓ Constant/zero-variance analysis completed.


In [13]:
# ==============================================================================
# 13. IDENTIFIER CANDIDATE DETECTION
# ==============================================================================

print("=" * 100)
print("13. IDENTIFIER CANDIDATE DETECTION")
print("=" * 100)

IDENTIFIER_REPORTS = []

ID_NAME_PATTERNS = [
    "id",
    "identifier",
    "index",
    "uuid",
    "record",
    "account",
    "customer",
    "patient",
    "member",
    "transaction",
    "employee",
]

for dataset_id, df in RAW_DATASETS.items():

    n_rows = len(df)

    for column in df.columns:

        series = df[column]

        name = str(column).strip().lower()

        unique_count = int(
            series.nunique(
                dropna=True
            )
        )

        unique_rate = (
            unique_count / n_rows
            if n_rows > 0
            else np.nan
        )

        name_based_signal = any(
            pattern in name
            for pattern in ID_NAME_PATTERNS
        )

        high_cardinality_signal = (
            pd.notna(unique_rate)
            and unique_rate >= 0.95
        )

        is_candidate = (
            name_based_signal
            or high_cardinality_signal
        )

        IDENTIFIER_REPORTS.append({
            "dataset_id": dataset_id,
            "column": str(column),
            "dtype": str(series.dtype),
            "n_unique": unique_count,
            "unique_rate": float(unique_rate),
            "name_based_signal": bool(
                name_based_signal
            ),
            "high_cardinality_signal": bool(
                high_cardinality_signal
            ),
            "identifier_candidate": bool(
                is_candidate
            ),
        })

IDENTIFIER_REPORT_DF = pd.DataFrame(
    IDENTIFIER_REPORTS
)

print(
    "\nIdentifier candidates:",
    int(
        IDENTIFIER_REPORT_DF[
            "identifier_candidate"
        ].sum()
    )
)

print(
    "\nIMPORTANT:"
    "\nIdentifier candidates are flagged only."
    "\nNo raw feature is removed in Notebook 01."
)

print("\n✓ Identifier candidate detection completed.")

13. IDENTIFIER CANDIDATE DETECTION

Identifier candidates: 20

IMPORTANT:
Identifier candidates are flagged only.
No raw feature is removed in Notebook 01.

✓ Identifier candidate detection completed.


In [16]:
# ==============================================================================
# 14. NUMERIC / CATEGORICAL INVENTORY
# ==============================================================================

print("=" * 100)
print("14. NUMERIC / CATEGORICAL INVENTORY")
print("=" * 100)

FEATURE_INVENTORY = []

for dataset_id, df in RAW_DATASETS.items():

    target_column = TARGET_COLUMNS.get(
        dataset_id
    )

    for position, column in enumerate(
        df.columns,
        start=1
    ):

        series = df[column]

        is_numeric = pd.api.types.is_numeric_dtype(
            series
        )

        is_bool = pd.api.types.is_bool_dtype(
            series
        )

        is_datetime = pd.api.types.is_datetime64_any_dtype(
            series
        )

        if is_bool:
            feature_type = "boolean"

        elif is_datetime:
            feature_type = "datetime"

        elif is_numeric:
            feature_type = "numeric"

        else:
            feature_type = "categorical"

        n_unique = int(
            series.nunique(
                dropna=True
            )
        )

        missing_count = int(
            series.isna().sum()
        )

        FEATURE_INVENTORY.append({

            "dataset_id": dataset_id,

            "column_position": int(
                position
            ),

            "column": str(column),

            "dtype": str(
                series.dtype
            ),

            "feature_type": feature_type,

            "is_numeric": bool(
                is_numeric
            ),

            "is_categorical": bool(
                feature_type == "categorical"
            ),

            "is_boolean": bool(
                is_bool
            ),

            "is_datetime": bool(
                is_datetime
            ),

            "is_target": bool(
                column == target_column
            ),

            "n_unique": n_unique,

            "missing_count": missing_count,

            "missing_rate": float(
                missing_count / len(df)
                if len(df) > 0
                else np.nan
            ),
        })

FEATURE_INVENTORY_DF = pd.DataFrame(
    FEATURE_INVENTORY
)

print(
    "\nTotal features:",
    len(FEATURE_INVENTORY_DF)
)

print(
    "Numeric features:",
    int(
        FEATURE_INVENTORY_DF[
            "is_numeric"
        ].sum()
    )
)

print(
    "Categorical features:",
    int(
        FEATURE_INVENTORY_DF[
            "is_categorical"
        ].sum()
    )
)

print("\n✓ Feature inventory completed.")

14. NUMERIC / CATEGORICAL INVENTORY

Total features: 82
Numeric features: 26
Categorical features: 56

✓ Feature inventory completed.


In [17]:
# ==============================================================================
# 15. RAW DATASET STATISTICAL SUMMARY
# ==============================================================================

print("=" * 100)
print("15. RAW DATASET STATISTICAL SUMMARY")
print("=" * 100)

RAW_STATISTICAL_SUMMARIES = {}

for dataset_id, df in RAW_DATASETS.items():

    print(
        f"\nGenerating summary: {dataset_id}"
    )

    summary = df.describe(
        include="all"
    ).transpose()

    summary.insert(
        0,
        "dataset_id",
        dataset_id
    )

    summary.insert(
        1,
        "column",
        summary.index.astype(str)
    )

    summary.reset_index(
        drop=True,
        inplace=True
    )

    RAW_STATISTICAL_SUMMARIES[
        dataset_id
    ] = summary

    print(
        f"  ✓ {summary.shape[0]} feature summaries"
    )

print("\n✓ Raw statistical summaries generated.")

15. RAW DATASET STATISTICAL SUMMARY

Generating summary: adult_income
  ✓ 15 feature summaries

Generating summary: bank_marketing
  ✓ 17 feature summaries

Generating summary: diabetes_130us
  ✓ 50 feature summaries

✓ Raw statistical summaries generated.


In [18]:
# ==============================================================================
# 16. RAW FILE SHA-256 FINGERPRINTING
# ==============================================================================

print("=" * 100)
print("16. RAW FILE SHA-256 FINGERPRINTING")
print("=" * 100)

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """
    Calculate SHA-256 fingerprint using chunked file reading.
    """

    file_path = Path(file_path)

    sha256 = hashlib.sha256()

    with open(file_path, "rb") as file:

        while True:

            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


FINGERPRINT_RECORDS = []

for dataset_id, file_path in RAW_DATASET_PATHS.items():

    file_path = Path(file_path)

    sha256 = calculate_sha256(
        file_path
    )

    stat = file_path.stat()

    record = {

        "dataset_id": dataset_id,

        "file_name": file_path.name,

        "file_path": str(
            file_path
        ),

        "file_extension":
            file_path.suffix.lower(),

        "file_size_bytes":
            int(stat.st_size),

        "modified_time":
            datetime.fromtimestamp(
                stat.st_mtime,
                tz=timezone.utc
            ).isoformat(),

        "sha256":
            sha256,

        "fingerprint_algorithm":
            "SHA-256",

    }

    FINGERPRINT_RECORDS.append(
        record
    )

    print(
        f"\n{dataset_id}"
    )

    print(
        f"  File   : {file_path.name}"
    )

    print(
        f"  Size   : {stat.st_size:,} bytes"
    )

    print(
        f"  SHA256 : {sha256}"
    )

FINGERPRINT_MANIFEST_DF = pd.DataFrame(
    FINGERPRINT_RECORDS
)

print("\n✓ SHA-256 fingerprinting completed.")

16. RAW FILE SHA-256 FINGERPRINTING

adult_income
  File   : adult_income.csv
  Size   : 5,271,057 bytes
  SHA256 : 9479f8b76861e48c66836d97937f2c5a469581672e12c795d770b297817ae3a1

bank_marketing
  File   : bank_marketing.csv
  Size   : 4,610,348 bytes
  SHA256 : d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d

diabetes_130us
  File   : diabetes_130us.csv
  Size   : 19,159,383 bytes
  SHA256 : 0689e7ec031237dc63031b938805c48377748761a3b26acab621567afa24df97

✓ SHA-256 fingerprinting completed.


In [19]:
# ==================================================================================================
# 17. DATASET VALIDATION ENGINE
# ==================================================================================================

print("=" * 100)
print("17. DATASET VALIDATION ENGINE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 17.1 Required objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "DATASET_IDS",
    "RAW_DATASETS",
    "RAW_DATASET_PATHS",
    "TARGET_COLUMNS",
]


_missing_objects = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if _missing_objects:

    raise RuntimeError(
        "Required objects are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in _missing_objects
        )
        + "\n\nExecute the preceding Notebook 01 sections first."
    )


# --------------------------------------------------------------------------------------------------
# 17.2 Validation configuration
# --------------------------------------------------------------------------------------------------

VALIDATION_RESULTS = []

VALIDATION_ERRORS = []


# --------------------------------------------------------------------------------------------------
# 17.3 Validate each dataset
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    print()
    print("-" * 100)
    print(f"Dataset: {dataset_id}")
    print("-" * 100)

    validation_pass = True
    dataset_errors = []

    # ----------------------------------------------------------------------------------------------
    # Dataset loaded
    # ----------------------------------------------------------------------------------------------

    if dataset_id not in RAW_DATASETS:

        validation_pass = False

        dataset_errors.append(
            "Dataset is not present in RAW_DATASETS."
        )

        df = None

    else:

        df = RAW_DATASETS[dataset_id]


    # ----------------------------------------------------------------------------------------------
    # Raw file validation
    # ----------------------------------------------------------------------------------------------

    raw_file = RAW_DATASET_PATHS.get(
        dataset_id
    )

    raw_file_exists = (
        raw_file is not None
        and Path(raw_file).exists()
    )

    if not raw_file_exists:

        validation_pass = False

        dataset_errors.append(
            "Raw dataset file does not exist."
        )


    # ----------------------------------------------------------------------------------------------
    # DataFrame validation
    # ----------------------------------------------------------------------------------------------

    dataframe_valid = isinstance(
        df,
        pd.DataFrame
    )

    if not dataframe_valid:

        validation_pass = False

        dataset_errors.append(
            "Loaded dataset is not a pandas DataFrame."
        )


    # ----------------------------------------------------------------------------------------------
    # Shape validation
    # ----------------------------------------------------------------------------------------------

    if dataframe_valid:

        n_rows = int(df.shape[0])
        n_columns = int(df.shape[1])

    else:

        n_rows = 0
        n_columns = 0


    if n_rows == 0:

        validation_pass = False

        dataset_errors.append(
            "Dataset contains zero rows."
        )


    if n_columns == 0:

        validation_pass = False

        dataset_errors.append(
            "Dataset contains zero columns."
        )


    # ----------------------------------------------------------------------------------------------
    # Column-name validation
    # ----------------------------------------------------------------------------------------------

    duplicate_column_names = False
    null_column_names = False

    if dataframe_valid:

        duplicate_column_names = bool(
            df.columns.duplicated().any()
        )

        null_column_names = bool(
            pd.isna(df.columns).any()
        )


        if duplicate_column_names:

            validation_pass = False

            dataset_errors.append(
                "Duplicate column names detected."
            )


        if null_column_names:

            validation_pass = False

            dataset_errors.append(
                "Null column names detected."
            )


    # ----------------------------------------------------------------------------------------------
    # Target validation
    # ----------------------------------------------------------------------------------------------

    target_column = TARGET_COLUMNS.get(
        dataset_id
    )

    target_exists = (
        dataframe_valid
        and target_column is not None
        and target_column in df.columns
    )


    if not target_exists:

        validation_pass = False

        if target_column is None:

            dataset_errors.append(
                "Target column is not configured."
            )

        else:

            dataset_errors.append(
                f"Configured target '{target_column}' "
                f"is not present in the raw dataset."
            )


    # ----------------------------------------------------------------------------------------------
    # Target statistics
    # ----------------------------------------------------------------------------------------------

    if target_exists:

        target_series = df[target_column]

        target_missing_count = int(
            target_series.isna().sum()
        )

        target_unique_count = int(
            target_series.nunique(dropna=True)
        )

        target_dtype = str(
            target_series.dtype
        )

        target_missing_rate = (
            target_missing_count / n_rows
            if n_rows > 0
            else 0.0
        )

    else:

        target_missing_count = None
        target_unique_count = None
        target_dtype = None
        target_missing_rate = None


    # ----------------------------------------------------------------------------------------------
    # Duplicate row validation
    # ----------------------------------------------------------------------------------------------

    duplicate_rows = None
    duplicate_row_rate = None

    if dataframe_valid:

        duplicate_rows = int(
            df.duplicated().sum()
        )

        duplicate_row_rate = (
            duplicate_rows / n_rows
            if n_rows > 0
            else 0.0
        )


    # ----------------------------------------------------------------------------------------------
    # Missing-value validation
    # ----------------------------------------------------------------------------------------------

    if dataframe_valid:

        total_missing_values = int(
            df.isna().sum().sum()
        )

        columns_with_missing_values = int(
            df.isna().any().sum()
        )

    else:

        total_missing_values = None
        columns_with_missing_values = None


    # ----------------------------------------------------------------------------------------------
    # Constant-feature validation
    # ----------------------------------------------------------------------------------------------

    if dataframe_valid:

        constant_columns = [
            column
            for column in df.columns
            if df[column].nunique(dropna=False) <= 1
        ]

        constant_feature_count = len(
            constant_columns
        )

    else:

        constant_columns = []
        constant_feature_count = None


    # ----------------------------------------------------------------------------------------------
    # Validation result
    # ----------------------------------------------------------------------------------------------

    validation_status = (
        "PASS"
        if validation_pass
        else "FAIL"
    )


    # ----------------------------------------------------------------------------------------------
    # Store result
    # ----------------------------------------------------------------------------------------------

    validation_record = {
        "dataset_id": dataset_id,
        "raw_file": str(raw_file) if raw_file is not None else None,

        "rows": n_rows,
        "columns": n_columns,

        "dataframe_valid": dataframe_valid,
        "raw_file_exists": raw_file_exists,

        "duplicate_column_names": duplicate_column_names,
        "null_column_names": null_column_names,

        "target_column": target_column,
        "target_exists": target_exists,
        "target_dtype": target_dtype,
        "target_unique_values": target_unique_count,
        "target_missing_count": target_missing_count,
        "target_missing_rate": target_missing_rate,

        "total_missing_values": total_missing_values,
        "columns_with_missing_values": columns_with_missing_values,

        "duplicate_rows": duplicate_rows,
        "duplicate_row_rate": duplicate_row_rate,

        "constant_feature_count": constant_feature_count,

        "validation_status": validation_status,
        "validation_errors": " | ".join(dataset_errors),
    }

    VALIDATION_RESULTS.append(
        validation_record
    )


    # ----------------------------------------------------------------------------------------------
    # Store errors
    # ----------------------------------------------------------------------------------------------

    if dataset_errors:

        VALIDATION_ERRORS.extend(
            [
                {
                    "dataset_id": dataset_id,
                    "error": error,
                }
                for error in dataset_errors
            ]
        )


    # ----------------------------------------------------------------------------------------------
    # Console output
    # ----------------------------------------------------------------------------------------------

    print(
        f"  Validation : {validation_status}"
    )

    print(
        f"  Shape      : {n_rows:,} × {n_columns:,}"
    )

    print(
        f"  Target     : {target_column!r}"
        if target_exists
        else
        f"  Target     : {target_column!r} [NOT VERIFIED]"
    )

    print(
        f"  Target OK  : {target_exists}"
    )

    if target_exists:

        print(
            f"  Target dtype       : {target_dtype}"
        )

        print(
            f"  Target unique      : {target_unique_count:,}"
        )

        print(
            f"  Target missing     : {target_missing_count:,}"
        )

    print(
        f"  Missing cells      : {total_missing_values:,}"
        if total_missing_values is not None
        else
        "  Missing cells      : None"
    )

    print(
        f"  Duplicate rows     : {duplicate_rows:,}"
        if duplicate_rows is not None
        else
        "  Duplicate rows     : None"
    )

    print(
        f"  Constant features  : {constant_feature_count:,}"
        if constant_feature_count is not None
        else
        "  Constant features  : None"
    )

    if dataset_errors:

        print()
        print("  Validation errors:")

        for error in dataset_errors:

            print(
                f"    ✗ {error}"
            )


# --------------------------------------------------------------------------------------------------
# 17.4 Convert validation results to DataFrame
# --------------------------------------------------------------------------------------------------

DATASET_VALIDATION_DF = pd.DataFrame(
    VALIDATION_RESULTS
)


# --------------------------------------------------------------------------------------------------
# 17.5 Global validation checks
# --------------------------------------------------------------------------------------------------

if len(DATASET_VALIDATION_DF) != len(DATASET_IDS):

    raise RuntimeError(
        "Validation result count does not match DATASET_IDS."
    )


if not DATASET_VALIDATION_DF["target_exists"].all():

    failed_targets = DATASET_VALIDATION_DF.loc[
        ~DATASET_VALIDATION_DF["target_exists"],
        "dataset_id"
    ].tolist()

    raise RuntimeError(
        "Target validation failed for:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in failed_targets
        )
    )


# --------------------------------------------------------------------------------------------------
# 17.6 Final dataset validation status
# --------------------------------------------------------------------------------------------------

FAILED_DATASET_VALIDATIONS = DATASET_VALIDATION_DF.loc[
    DATASET_VALIDATION_DF["validation_status"] != "PASS",
    "dataset_id"
].tolist()


if FAILED_DATASET_VALIDATIONS:

    print()
    print("=" * 100)
    print("DATASET VALIDATION FAILURE")
    print("=" * 100)

    for dataset_id in FAILED_DATASET_VALIDATIONS:

        print(
            f"✗ {dataset_id}"
        )

    raise RuntimeError(
        "One or more datasets failed validation."
    )


# --------------------------------------------------------------------------------------------------
# 17.7 Final output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("DATASET VALIDATION SUMMARY")
print("=" * 100)

print()

for _, row in DATASET_VALIDATION_DF.iterrows():

    print(
        f"✓ {row['dataset_id']:<20} "
        f"Validation: {row['validation_status']:<4} | "
        f"Shape: {int(row['rows']):,} × {int(row['columns']):,} | "
        f"Target: {row['target_column']}"
    )

print()
print(
    f"Datasets validated : {len(DATASET_VALIDATION_DF)}"
)

print(
    f"Datasets passed    : "
    f"{(DATASET_VALIDATION_DF['validation_status'] == 'PASS').sum()}"
)

print(
    f"Targets verified   : "
    f"{DATASET_VALIDATION_DF['target_exists'].sum()}"
)

print()
print("✓ All datasets passed structural and target validation.")
print("✓ Target metadata propagated from Section 9.")
print("✓ Dataset validation engine completed successfully.")

17. DATASET VALIDATION ENGINE

----------------------------------------------------------------------------------------------------
Dataset: adult_income
----------------------------------------------------------------------------------------------------
  Validation : PASS
  Shape      : 48,842 × 15
  Target     : 'income'
  Target OK  : True
  Target dtype       : object
  Target unique      : 2
  Target missing     : 0
  Missing cells      : 6,465
  Duplicate rows     : 52
  Constant features  : 0

----------------------------------------------------------------------------------------------------
Dataset: bank_marketing
----------------------------------------------------------------------------------------------------
  Validation : PASS
  Shape      : 45,211 × 17
  Target     : 'y'
  Target OK  : True
  Target dtype       : object
  Target unique      : 2
  Target missing     : 0
  Missing cells      : 0
  Duplicate rows     : 0
  Constant features  : 0

-------------------------

In [21]:
# ==============================================================================
# 18. VALIDATION SUMMARY
# ==============================================================================

print("=" * 100)
print("18. VALIDATION SUMMARY")
print("=" * 100)

validation_pass_count = int(
    (
        DATASET_VALIDATION_DF["validation_status"] == "PASS"
    ).sum()
)

validation_total = int(
    len(DATASET_VALIDATION_DF)
)

validation_failed_count = (
    validation_total
    - validation_pass_count
)

print(
    f"\nDatasets evaluated : {validation_total}"
)

print(
    f"Validation PASS    : {validation_pass_count}"
)

print(
    f"Validation FAIL    : {validation_failed_count}"
)

if validation_failed_count > 0:

    print("\n⚠ Failed datasets:")

    print(
        DATASET_VALIDATION_DF.loc[
            DATASET_VALIDATION_DF["validation_status"] != "PASS"
        ].to_string(index=False)
    )

else:

    print(
        "\n✓ All loaded datasets passed "
        "critical raw-data validation."
    )

18. VALIDATION SUMMARY

Datasets evaluated : 3
Validation PASS    : 3
Validation FAIL    : 0

✓ All loaded datasets passed critical raw-data validation.


In [22]:
# ==============================================================================
# 19. SAVE FEATURE INVENTORIES
# ==============================================================================

print("=" * 100)
print("19. SAVE FEATURE INVENTORIES")
print("=" * 100)

INVENTORY_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw_validation"
    / "feature_inventory"
)

INVENTORY_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_INVENTORY_PATH = (
    INVENTORY_DIR
    / "feature_inventory.csv"
)

FEATURE_INVENTORY_DF.to_csv(
    FEATURE_INVENTORY_PATH,
    index=False
)

print(
    f"\n✓ Saved:"
    f"\n  {FEATURE_INVENTORY_PATH}"
)

# ------------------------------------------------------------------------------
# Dataset-specific inventories
# ------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    subset = FEATURE_INVENTORY_DF[
        FEATURE_INVENTORY_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if subset.empty:
        continue

    path = (
        INVENTORY_DIR
        / f"{dataset_id}_feature_inventory.csv"
    )

    subset.to_csv(
        path,
        index=False
    )

print("\n✓ Feature inventories saved.")

19. SAVE FEATURE INVENTORIES

✓ Saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/feature_inventory/feature_inventory.csv

✓ Feature inventories saved.


In [23]:
# ==============================================================================
# 20. SAVE MISSINGNESS REPORTS
# ==============================================================================

print("=" * 100)
print("20. SAVE MISSINGNESS REPORTS")
print("=" * 100)

MISSINGNESS_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw_validation"
    / "missingness"
)

MISSINGNESS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MISSINGNESS_PATH = (
    MISSINGNESS_DIR
    / "raw_missingness_report.csv"
)

MISSINGNESS_REPORT_DF.to_csv(
    MISSINGNESS_PATH,
    index=False
)

print(
    f"\n✓ Saved:"
    f"\n  {MISSINGNESS_PATH}"
)

for dataset_id in DATASET_IDS:

    subset = MISSINGNESS_REPORT_DF[
        MISSINGNESS_REPORT_DF[
            "dataset_id"
        ] == dataset_id
    ]

    if subset.empty:
        continue

    path = (
        MISSINGNESS_DIR
        / f"{dataset_id}_missingness.csv"
    )

    subset.to_csv(
        path,
        index=False
    )

print("\n✓ Missingness reports saved.")

20. SAVE MISSINGNESS REPORTS

✓ Saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/missingness/raw_missingness_report.csv

✓ Missingness reports saved.


In [25]:
# ==============================================================================
# 21. SAVE STRUCTURAL REPORTS
# ==============================================================================

print("=" * 100)
print("21. SAVE STRUCTURAL REPORTS")
print("=" * 100)

STRUCTURAL_DIR = (
    PROJECT_ROOT
    / "results"
    / "raw_validation"
    / "structural"
)

STRUCTURAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STRUCTURAL_PATH = (
    STRUCTURAL_DIR
    / "structural_validation_report.csv"
)

STRUCTURAL_REPORT_DF.to_csv(
    STRUCTURAL_PATH,
    index=False
)

DUPLICATE_PATH = (
    STRUCTURAL_DIR
    / "duplicate_report.csv"
)

DUPLICATE_REPORT_DF.to_csv(
    DUPLICATE_PATH,
    index=False
)

CONSTANT_PATH = (
    STRUCTURAL_DIR
    / "constant_zero_variance_report.csv"
)

CONSTANT_REPORT_DF.to_csv(
    CONSTANT_PATH,
    index=False
)

IDENTIFIER_PATH = (
    STRUCTURAL_DIR
    / "identifier_candidate_report.csv"
)

IDENTIFIER_REPORT_DF.to_csv(
    IDENTIFIER_PATH,
    index=False
)

VALIDATION_PATH = (
    STRUCTURAL_DIR
    / "dataset_validation_report.csv"
)

DATASET_VALIDATION_DF.to_csv(
    VALIDATION_PATH,
    index=False
)

print("\nSaved structural reports:")

for path in [
    STRUCTURAL_PATH,
    DUPLICATE_PATH,
    CONSTANT_PATH,
    IDENTIFIER_PATH,
    VALIDATION_PATH,
]:

    print(f"  ✓ {path}")

print("\n✓ Structural reports saved.")

21. SAVE STRUCTURAL REPORTS

Saved structural reports:
  ✓ /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/structural_validation_report.csv
  ✓ /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/duplicate_report.csv
  ✓ /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/constant_zero_variance_report.csv
  ✓ /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/identifier_candidate_report.csv
  ✓ /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/structural/dataset_validation_report.csv

✓ Structural reports saved.


In [27]:
# ==============================================================================
# 22. BUILD VALIDATED DATASET REGISTRY
# ==============================================================================

print("=" * 100)
print("22. BUILD VALIDATED DATASET REGISTRY")
print("=" * 100)

VALIDATED_DATASET_REGISTRY = []

for dataset_id in DATASET_IDS:

    df = RAW_DATASETS.get(
        dataset_id
    )

    file_path = RAW_DATASET_PATHS.get(
        dataset_id
    )

    if df is None:
        continue

    # ------------------------------------------------------------------------------
    # Validation status
    # ------------------------------------------------------------------------------

    validation_row = (
        DATASET_VALIDATION_DF[
            DATASET_VALIDATION_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    if validation_row.empty:
        validation_pass = False
    else:
        validation_pass = (
            validation_row.iloc[0][
                "validation_status"
            ] == "PASS"
        )

    # ------------------------------------------------------------------------------
    # Feature inventory
    # ------------------------------------------------------------------------------

    feature_subset = (
        FEATURE_INVENTORY_DF[
            FEATURE_INVENTORY_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    numeric_count = int(
        feature_subset[
            "is_numeric"
        ].sum()
    )

    categorical_count = int(
        feature_subset[
            "is_categorical"
        ].sum()
    )

    # ------------------------------------------------------------------------------
    # Missingness
    # ------------------------------------------------------------------------------

    missing_count = int(
        MISSINGNESS_REPORT_DF[
            MISSINGNESS_REPORT_DF[
                "dataset_id"
            ] == dataset_id
        ]["missing_count"].sum()
    )

    # ------------------------------------------------------------------------------
    # Duplicate rows
    # ------------------------------------------------------------------------------

    duplicate_row_count = int(
        DUPLICATE_REPORT_DF[
            DUPLICATE_REPORT_DF[
                "dataset_id"
            ] == dataset_id
        ].iloc[0][
            "duplicate_rows_excluding_first"
        ]
    )

    # ------------------------------------------------------------------------------
    # Constant features
    # ------------------------------------------------------------------------------

    constant_count = int(
        CONSTANT_REPORT_DF[
            CONSTANT_REPORT_DF[
                "dataset_id"
            ] == dataset_id
        ]["is_constant"].sum()
    )

    # ------------------------------------------------------------------------------
    # Identifier candidates
    # ------------------------------------------------------------------------------

    identifier_count = int(
        IDENTIFIER_REPORT_DF[
            IDENTIFIER_REPORT_DF[
                "dataset_id"
            ] == dataset_id
        ]["identifier_candidate"].sum()
    )

    # ------------------------------------------------------------------------------
    # SHA-256 fingerprint
    # ------------------------------------------------------------------------------

    fingerprint_row = (
        FINGERPRINT_MANIFEST_DF[
            FINGERPRINT_MANIFEST_DF[
                "dataset_id"
            ] == dataset_id
        ]
    )

    sha256 = (
        fingerprint_row.iloc[0]["sha256"]
        if not fingerprint_row.empty
        else None
    )

    # ------------------------------------------------------------------------------
    # Registry record
    # ------------------------------------------------------------------------------

    registry_record = {

        "dataset_id":
            dataset_id,

        "raw_file":
            str(file_path)
            if file_path is not None
            else None,

        "raw_file_sha256":
            sha256,

        "n_rows":
            int(len(df)),

        "n_columns":
            int(len(df.columns)),

        "n_numeric_features":
            numeric_count,

        "n_categorical_features":
            categorical_count,

        "target_column":
            TARGET_COLUMNS.get(
                dataset_id
            ),

        "missing_cells":
            missing_count,

        "duplicate_rows":
            duplicate_row_count,

        "constant_features":
            constant_count,

        "identifier_candidates":
            identifier_count,

        "validation_pass":
            bool(validation_pass),

        "registry_created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    VALIDATED_DATASET_REGISTRY.append(
        registry_record
    )

# ------------------------------------------------------------------------------
# Convert to DataFrame
# ------------------------------------------------------------------------------

VALIDATED_DATASET_REGISTRY_DF = pd.DataFrame(
    VALIDATED_DATASET_REGISTRY
)

print(
    VALIDATED_DATASET_REGISTRY_DF.to_string(
        index=False
    )
)

print("\n✓ Validated dataset registry built.")

22. BUILD VALIDATED DATASET REGISTRY
    dataset_id                                                            raw_file                                                  raw_file_sha256  n_rows  n_columns  n_numeric_features  n_categorical_features target_column  missing_cells  duplicate_rows  constant_features  identifier_candidates  validation_pass             registry_created_utc
  adult_income   /content/drive/MyDrive/SPP_GAN_Research/data/raw/adult_income.csv 9479f8b76861e48c66836d97937f2c5a469581672e12c795d770b297817ae3a1   48842         15                   6                       9        income           6465              52                  0                      0             True 2026-09-09T11:25:07.326899+00:00
bank_marketing /content/drive/MyDrive/SPP_GAN_Research/data/raw/bank_marketing.csv d1513ec63b385506f7cfce9f2c5caa9fe99e7ba4e8c3fa264b3aaf0f849ed32d   45211         17                   7                      10             y              0               0            

In [28]:
# ==============================================================================
# 23. SAVE FINGERPRINT MANIFEST
# ==============================================================================

print("=" * 100)
print("23. SAVE FINGERPRINT MANIFEST")
print("=" * 100)

MANIFEST_DIR = (
    PROJECT_ROOT
    / "manifests"
)

MANIFEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINGERPRINT_MANIFEST_PATH = (
    MANIFEST_DIR
    / "raw_dataset_fingerprint_manifest.csv"
)

FINGERPRINT_MANIFEST_DF.to_csv(
    FINGERPRINT_MANIFEST_PATH,
    index=False
)

print(
    f"\n✓ Fingerprint manifest saved:"
    f"\n  {FINGERPRINT_MANIFEST_PATH}"
)

# JSON version for machine-readable provenance
FINGERPRINT_JSON_PATH = (
    MANIFEST_DIR
    / "raw_dataset_fingerprint_manifest.json"
)

with open(
    FINGERPRINT_JSON_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        FINGERPRINT_RECORDS,
        file,
        indent=2
    )

print(
    f"✓ JSON fingerprint manifest saved:"
    f"\n  {FINGERPRINT_JSON_PATH}"
)

23. SAVE FINGERPRINT MANIFEST

✓ Fingerprint manifest saved:
  /content/drive/MyDrive/SPP_GAN_Research/manifests/raw_dataset_fingerprint_manifest.csv
✓ JSON fingerprint manifest saved:
  /content/drive/MyDrive/SPP_GAN_Research/manifests/raw_dataset_fingerprint_manifest.json


In [29]:
# ==============================================================================
# 24. PROVENANCE MANIFEST
# ==============================================================================

print("=" * 100)
print("24. PROVENANCE MANIFEST")
print("=" * 100)

PROVENANCE_MANIFEST = {

    "project": {
        "project_name":
            "SPP-GAN",

        "project_root":
            str(PROJECT_ROOT),

        "notebook":
            "Notebook 01 — Raw Dataset Loading & Validation",
    },

    "experiment": {
        "purpose":
            "Raw dataset loading, validation, provenance and fingerprinting",

        "raw_data_modified":
            False,

        "preprocessing_applied":
            False,

        "encoding_applied":
            False,

        "scaling_applied":
            False,

        "imputation_applied":
            False,

        "synthetic_generation_applied":
            False,
    },

    "environment": {
        "python_version":
            sys.version,

        "platform":
            platform.platform(),

        "pandas_version":
            pd.__version__,

        "numpy_version":
            np.__version__,
    },

    "datasets": VALIDATED_DATASET_REGISTRY,

    "execution": {

        "execution_timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "dataset_count":
            len(DATASET_IDS),

        "loaded_dataset_count":
            len(RAW_DATASETS),

        "failed_dataset_count":
            len(RAW_DATASET_LOAD_ERRORS),
    },

    "integrity": {

        "hash_algorithm":
            "SHA-256",

        "raw_file_fingerprinting":
            True,

        "validation_engine":
            True,
    },
}

PROVENANCE_JSON_PATH = (
    MANIFEST_DIR
    / "notebook_01_provenance_manifest.json"
)

with open(
    PROVENANCE_JSON_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        PROVENANCE_MANIFEST,
        file,
        indent=2,
        default=str
    )

print(
    f"\n✓ Provenance manifest saved:"
    f"\n  {PROVENANCE_JSON_PATH}"
)

24. PROVENANCE MANIFEST

✓ Provenance manifest saved:
  /content/drive/MyDrive/SPP_GAN_Research/manifests/notebook_01_provenance_manifest.json


In [31]:
# ==============================================================================
# 25. FINAL INTEGRITY VERIFICATION
# ==============================================================================

print("=" * 100)
print("25. FINAL INTEGRITY VERIFICATION")
print("=" * 100)

INTEGRITY_CHECKS = {}

# ------------------------------------------------------------------------------
# Project
# ------------------------------------------------------------------------------

INTEGRITY_CHECKS[
    "project_root_exists"
] = PROJECT_ROOT.exists()

INTEGRITY_CHECKS[
    "raw_data_directory_exists"
] = RAW_DATA_DIR.exists()

# ------------------------------------------------------------------------------
# Dataset count
# ------------------------------------------------------------------------------

INTEGRITY_CHECKS[
    "all_requested_datasets_loaded"
] = (
    len(RAW_DATASETS)
    == len(DATASET_IDS)
)

# ------------------------------------------------------------------------------
# Validation
# ------------------------------------------------------------------------------

if not DATASET_VALIDATION_DF.empty:

    INTEGRITY_CHECKS[
        "all_datasets_pass_validation"
    ] = bool(
        (
            DATASET_VALIDATION_DF[
                "validation_status"
            ]
            == "PASS"
        ).all()
    )

else:

    INTEGRITY_CHECKS[
        "all_datasets_pass_validation"
    ] = False

# ------------------------------------------------------------------------------
# Fingerprints
# ------------------------------------------------------------------------------

INTEGRITY_CHECKS[
    "all_loaded_files_fingerprinted"
] = (
    len(FINGERPRINT_MANIFEST_DF)
    == len(RAW_DATASETS)
)

# ------------------------------------------------------------------------------
# Registry
# ------------------------------------------------------------------------------

INTEGRITY_CHECKS[
    "validated_registry_complete"
] = (
    len(VALIDATED_DATASET_REGISTRY_DF)
    == len(RAW_DATASETS)
)

# ------------------------------------------------------------------------------
# Reports
# ------------------------------------------------------------------------------

INTEGRITY_CHECKS[
    "feature_inventory_created"
] = (
    not FEATURE_INVENTORY_DF.empty
)

INTEGRITY_CHECKS[
    "missingness_report_created"
] = (
    not MISSINGNESS_REPORT_DF.empty
)

INTEGRITY_CHECKS[
    "structural_report_created"
] = (
    not STRUCTURAL_REPORT_DF.empty
)

# ------------------------------------------------------------------------------
# Manifest files
# ------------------------------------------------------------------------------

INTEGRITY_CHECKS[
    "fingerprint_manifest_exists"
] = FINGERPRINT_MANIFEST_PATH.exists()

INTEGRITY_CHECKS[
    "provenance_manifest_exists"
] = PROVENANCE_JSON_PATH.exists()

# ------------------------------------------------------------------------------
# Print
# ------------------------------------------------------------------------------

print("\nIntegrity checks:")

for check_name, passed in INTEGRITY_CHECKS.items():

    print(
        f"  {'PASS' if passed else 'FAIL':<6} "
        f"{check_name}"
    )

FINAL_INTEGRITY_PASS = all(
    INTEGRITY_CHECKS.values()
)

print("\n" + "-" * 100)

print(
    "FINAL INTEGRITY STATUS:",
    "PASS" if FINAL_INTEGRITY_PASS else "FAIL"
)

if not FINAL_INTEGRITY_PASS:

    failed_checks = [
        name
        for name, passed
        in INTEGRITY_CHECKS.items()
        if not passed
    ]

    print("\nFailed checks:")

    for check in failed_checks:
        print(f"  ✗ {check}")

    raise RuntimeError(
        "Notebook 01 final integrity verification FAILED."
    )

print(
    "\n✓ Notebook 01 integrity verification passed."
)

25. FINAL INTEGRITY VERIFICATION

Integrity checks:
  PASS   project_root_exists
  PASS   raw_data_directory_exists
  PASS   all_requested_datasets_loaded
  PASS   all_datasets_pass_validation
  PASS   all_loaded_files_fingerprinted
  PASS   validated_registry_complete
  PASS   feature_inventory_created
  PASS   missingness_report_created
  PASS   structural_report_created
  PASS   fingerprint_manifest_exists
  PASS   provenance_manifest_exists

----------------------------------------------------------------------------------------------------
FINAL INTEGRITY STATUS: PASS

✓ Notebook 01 integrity verification passed.


In [34]:
# ==============================================================================
# 26. COMPLETION SUMMARY
# ==============================================================================

print("\n")
print("=" * 100)
print("NOTEBOOK 01 — COMPLETION SUMMARY")
print("=" * 100)

print("\nRAW DATASET FOUNDATION")
print("-" * 100)

print(
    f"Project Root              : {PROJECT_ROOT}"
)

print(
    f"Datasets Requested        : {len(DATASET_IDS)}"
)

print(
    f"Datasets Loaded           : {len(RAW_DATASETS)}"
)

print(
    f"Datasets Failed           : "
    f"{len(RAW_DATASET_LOAD_ERRORS)}"
)

print(
    f"Total Features            : "
    f"{len(FEATURE_INVENTORY_DF)}"
)

print(
    f"Numeric Features          : "
    f"{FEATURE_INVENTORY_DF['is_numeric'].sum()}"
)

print(
    f"Categorical Features      : "
    f"{FEATURE_INVENTORY_DF['is_categorical'].sum()}"
)

print(
    f"Columns With Missing Data : "
    f"{(MISSINGNESS_REPORT_DF['missing_count'] > 0).sum()}"
)

print(
    f"Duplicate Rows Detected   : "
    f"{DUPLICATE_REPORT_DF['duplicate_rows_excluding_first'].sum()}"
)

print(
    f"Constant Features         : "
    f"{CONSTANT_REPORT_DF['is_constant'].sum()}"
)

print(
    f"Identifier Candidates     : "
    f"{IDENTIFIER_REPORT_DF['identifier_candidate'].sum()}"
)

print("\nVALIDATION")
print("-" * 100)

print(
    f"Validation Status         : "
    f"{'PASS' if FINAL_INTEGRITY_PASS else 'FAIL'}"
)

print(
    f"Fingerprinting            : SHA-256"
)

print(
    f"Raw Data Modified         : NO"
)

print("\nPRIMARY ARTIFACTS")
print("-" * 100)

print(
    f"Feature Inventory         : "
    f"{FEATURE_INVENTORY_PATH}"
)

print(
    f"Missingness Report        : "
    f"{MISSINGNESS_PATH}"
)

print(
    f"Structural Report         : "
    f"{STRUCTURAL_PATH}"
)

print(
    f"Duplicate Report          : "
    f"{DUPLICATE_PATH}"
)

print(
    f"Constant Report           : "
    f"{CONSTANT_PATH}"
)

print(
    f"Identifier Report         : "
    f"{IDENTIFIER_PATH}"
)

print(
    f"Fingerprint Manifest      : "
    f"{FINGERPRINT_MANIFEST_PATH}"
)

print(
    f"Provenance Manifest       : "
    f"{PROVENANCE_JSON_PATH}"
)

print("\n" + "=" * 100)
print("NOTEBOOK 01 COMPLETE — RAW DATA FOUNDATION ESTABLISHED")
print("=" * 100)

print(
    "\nDownstream notebooks may now consume "
    "VALIDATED_DATASET_REGISTRY_DF and the "
    "raw dataset artifacts."
)



NOTEBOOK 01 — COMPLETION SUMMARY

RAW DATASET FOUNDATION
----------------------------------------------------------------------------------------------------
Project Root              : /content/drive/MyDrive/SPP_GAN_Research
Datasets Requested        : 3
Datasets Loaded           : 3
Datasets Failed           : 0
Total Features            : 82
Numeric Features          : 26
Categorical Features      : 56
Columns With Missing Data : 5
Duplicate Rows Detected   : 52
Constant Features         : 2
Identifier Candidates     : 20

VALIDATION
----------------------------------------------------------------------------------------------------
Validation Status         : PASS
Fingerprinting            : SHA-256
Raw Data Modified         : NO

PRIMARY ARTIFACTS
----------------------------------------------------------------------------------------------------
Feature Inventory         : /content/drive/MyDrive/SPP_GAN_Research/results/raw_validation/feature_inventory/feature_inventory.csv
Mis